# India districts: coordinates, road-distance matrix, cleaning, export
Run cells top to bottom. Set `USER_AGENT` (real email) in the config cell first.

`pip install requests beautifulsoup4 lxml pandas numpy geopy tqdm openpyxl`

In [ ]:
import io
import os
import re
import time

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from geopy.exc import GeocoderInsufficientPrivileges, GeocoderServiceError
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim
from openpyxl import Workbook
from openpyxl.styles import Font
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

# ---- config ----
USER_AGENT = "districts-dataset-mso-project/1.0 (amit_jg@cs.iitr.ac.in)"  # unique + real email, else Nominatim 403s

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_districts_in_India"
COORDS_CSV = "districts_coordinates.csv"
CHECKPOINT_CSV = "districts_checkpoint.csv"
CHECKPOINT_EVERY = 50
LAT_RANGE, LON_RANGE = (6.0, 38.0), (68.0, 98.0)  # rough India bounding box

OSRM_URL = "http://router.project-osrm.org/table/v1/driving/"
BLOCK = 50          # sources + destinations per request = 2*BLOCK <= 100 (public server limit)
DELAY = 1.0         # seconds between OSRM requests
MAX_ATTEMPTS = 5
CACHE_DIR = "osrm_cache"
RAW_MATRIX_CSV = "indian_districts_road_distance_matrix.csv"

CIRCUITY = 1.3
EARTH_RADIUS_KM = 6371.0088
ISLAND_STATES = ("Andaman and Nicobar", "Lakshadweep")  # no road link to the mainland

INDIA_AREA_KM2 = 3_287_263   # official total area of India
INTRA_DISTRICT_KM = None     # diagonal value; None -> computed from average district area

OUT_XLSX = "districts.xlsx"
OUT_MAP_CSV = "district_id_map.csv"

## 1. Scrape districts

In [ ]:
def clean_text(text) -> str:
    """Strip footnote markers like [1] and collapse whitespace."""
    text = re.sub(r"\[.*?\]", "", str(text))
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def scrape_districts() -> pd.DataFrame:
    """Wikipedia -> DataFrame[district_id, district_name, state, headquarters]."""
    resp = requests.get(WIKI_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    rows = []
    for table in soup.select("table.wikitable"):
        heading = table.find_previous(["h2", "h3"])
        state = clean_text(heading.get_text()) if heading else ""
        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
        except ValueError:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [" ".join(map(str, c)).strip() for c in df.columns]
        df.columns = [clean_text(c) for c in df.columns]

        dist_col = next(
            (c for c in df.columns
             if re.match(r"(?i)^districts?\b", c) and not re.search(r"(?i)number|no\.", c)),
            None,
        )
        if dist_col is None:
            continue
        hq_col = next((c for c in df.columns if re.search(r"(?i)headquarter|capital", c)), None)

        for _, r in df.iterrows():
            name = clean_text(r[dist_col])
            if (not name or name.lower() in {"nan", "overview", "total", "district", "districts"}
                    or name.isdigit()):
                continue
            hq = clean_text(r[hq_col]) if hq_col and pd.notna(r[hq_col]) else ""
            rows.append({"district_name": name, "state": state, "headquarters": hq})

    df = pd.DataFrame(rows).drop_duplicates(subset=["district_name", "state"]).reset_index(drop=True)
    df.insert(0, "district_id", range(1, len(df) + 1))
    return df

## 2. Geocode (Nominatim)

In [ ]:
def geocode_districts(districts: pd.DataFrame) -> pd.DataFrame:
    """Add latitude/longitude via Nominatim (HQ query, then district fallback).
    Checkpoints every CHECKPOINT_EVERY districts; reruns resume. Writes COORDS_CSV."""
    if "YOUR_REAL_EMAIL" in USER_AGENT or "example.com" in USER_AGENT:
        raise SystemExit("Set USER_AGENT to a unique string with your real contact email.")

    done = {}
    if os.path.exists(CHECKPOINT_CSV):
        ck = pd.read_csv(CHECKPOINT_CSV).dropna(subset=["latitude", "longitude"])
        done = {int(r.district_id): (r.latitude, r.longitude) for r in ck.itertuples()}
        print(f"Resuming: {len(done)} districts already processed")

    geolocator = Nominatim(user_agent=USER_AGENT, timeout=10)
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1.1,  # Nominatim policy: max 1 req/s
        max_retries=2,
        error_wait_seconds=10,
        swallow_exceptions=False,
    )
    ERR = object()  # transient failure: not recorded as "not found"

    def in_india(loc):
        return (loc is not None
                and LAT_RANGE[0] <= loc.latitude <= LAT_RANGE[1]
                and LON_RANGE[0] <= loc.longitude <= LON_RANGE[1])

    def lookup(query):
        try:
            loc = geocode(query, country_codes="in")
        except GeocoderInsufficientPrivileges:
            raise SystemExit("HTTP 403 from Nominatim: blocked user-agent/IP. Fix USER_AGENT, wait, retry.")
        except GeocoderServiceError:
            return ERR
        return loc if in_india(loc) else None

    results = dict(done)

    def save(path):
        out = districts[["district_id", "district_name", "state"]].copy()
        out["latitude"] = out["district_id"].map(lambda i: results.get(i, (None, None))[0])
        out["longitude"] = out["district_id"].map(lambda i: results.get(i, (None, None))[1])
        out.to_csv(path, index=False)
        return out

    since_ckpt = 0
    try:
        for r in tqdm(districts.itertuples(), total=len(districts), desc="Geocoding"):
            if r.district_id in results:
                continue
            loc = None
            if r.headquarters:
                loc = lookup(f"{r.headquarters}, {r.district_name}, {r.state}, India")
            if loc is None or loc is ERR:  # fallback
                loc = lookup(f"{r.district_name}, {r.state}, India")
            if loc is ERR:
                continue  # retried on next run
            results[r.district_id] = (loc.latitude, loc.longitude) if loc else (None, None)
            since_ckpt += 1
            if since_ckpt >= CHECKPOINT_EVERY:
                save(CHECKPOINT_CSV)
                since_ckpt = 0
    except KeyboardInterrupt:
        print("Interrupted; saving checkpoint.")
    finally:
        save(CHECKPOINT_CSV)

    final = save(COORDS_CSV)
    print(f"Wrote {COORDS_CSV}: {len(final)} rows, {int(final['latitude'].isna().sum())} without coordinates")
    return final

## 3. OSRM road-distance matrix

In [ ]:
def make_session() -> requests.Session:
    """requests session with retry/backoff on 429 and 5xx."""
    s = requests.Session()
    retry = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"], respect_retry_after_header=True)
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.headers["User-Agent"] = USER_AGENT
    return s

In [ ]:
def fetch_block(session, src, dst, same: bool) -> np.ndarray:
    """One OSRM Table request. src/dst are lists of (lat, lon). Returns meters; NaN = no route."""
    def fmt(points):  # OSRM wants lon,lat separated by ';'
        return ";".join(f"{lon:.6f},{lat:.6f}" for lat, lon in points)

    if same:
        coords, query = src, "annotations=distance"
    else:
        coords = src + dst
        s_idx = ";".join(map(str, range(len(src))))
        d_idx = ";".join(map(str, range(len(src), len(coords))))
        query = f"annotations=distance&sources={s_idx}&destinations={d_idx}"
    url = f"{OSRM_URL}{fmt(coords)}?{query}"  # built manually: requests would encode ';'

    last_err = None
    for attempt in range(MAX_ATTEMPTS):
        try:
            r = session.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()
            if data.get("code") != "Ok":
                raise RuntimeError(f"OSRM code={data.get('code')}: {data.get('message')}")
            return np.array(data["distances"], dtype=float)  # null -> NaN
        except (requests.RequestException, ValueError, RuntimeError, KeyError) as e:
            last_err = e
            time.sleep(DELAY * 2 ** (attempt + 1))
    raise RuntimeError(f"Block failed after {MAX_ATTEMPTS} attempts: {last_err}")

In [ ]:
def build_osrm_matrix(coords: pd.DataFrame) -> pd.DataFrame:
    """NxN road-distance matrix in km, indexed by district_id. Blocks are cached in CACHE_DIR."""
    if "YOUR_REAL_EMAIL" in USER_AGENT:
        raise SystemExit("Set USER_AGENT to include your real contact email.")

    df = coords.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    dropped = len(coords) - len(df)
    if dropped:
        print(f"Dropping {dropped} districts without coordinates")

    n = len(df)
    pts = list(zip(df["latitude"], df["longitude"]))
    starts = list(range(0, n, BLOCK))
    os.makedirs(CACHE_DIR, exist_ok=True)
    session = make_session()
    M = np.full((n, n), np.nan)

    for i in tqdm(starts, desc="OSRM block rows"):
        for j in starts:
            path = os.path.join(CACHE_DIR, f"{i}_{j}.npy")
            if os.path.exists(path):
                block = np.load(path)
            else:
                block = fetch_block(session, pts[i:i + BLOCK], pts[j:j + BLOCK], same=(i == j))
                np.save(path, block)
                time.sleep(DELAY)
            M[i:i + block.shape[0], j:j + block.shape[1]] = block

    M = np.round(M / 1000.0, 3)  # meters -> km
    ids = df["district_id"].astype(int)
    raw = pd.DataFrame(M, index=ids, columns=ids)
    raw.index.name = "district_id"
    raw.to_csv(RAW_MATRIX_CSV)
    print(f"Wrote {RAW_MATRIX_CSV}: {raw.shape}, {int(np.isnan(M).sum())} unroutable pairs")
    return raw

## 4. Clean and validate

In [ ]:
def haversine_matrix(lat_deg, lon_deg) -> np.ndarray:
    """Pairwise great-circle distances in km."""
    lat, lon = np.radians(np.asarray(lat_deg, float)), np.radians(np.asarray(lon_deg, float))
    a = (np.sin((lat[None, :] - lat[:, None]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin((lon[None, :] - lon[:, None]) / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

In [ ]:
def clean_matrix(raw: pd.DataFrame, coords: pd.DataFrame) -> pd.DataFrame:
    """Fix NaN / inf / negative / off-diagonal-zero entries, zero the diagonal, make symmetric.
    Fill order: reverse direction (averaged if both valid), then Haversine x CIRCUITY."""
    m = raw.apply(pd.to_numeric, errors="coerce")
    m.columns = m.columns.astype(int)
    m.index = m.index.astype(int)
    assert m.shape[0] == m.shape[1] and (m.index == m.columns).all(), "Matrix must be square with matching labels"

    c = coords.set_index("district_id").loc[m.index]
    if c[["latitude", "longitude"]].isna().any().any():
        raise SystemExit("Some matrix districts lack coordinates; cannot compute fallback distances")

    A = m.to_numpy(float).copy()
    n = len(A)
    eye = np.eye(n, dtype=bool)

    invalid = (~np.isfinite(A) | (A < 0) | ((A == 0) & ~eye)) & ~eye
    A[invalid] = np.nan
    np.fill_diagonal(A, 0.0)

    At = A.T
    S = np.where(np.isnan(A), At, np.where(np.isnan(At), A, (A + At) / 2))
    n_mirror = int((np.isnan(A) & ~np.isnan(At)).sum())

    still = np.isnan(S)
    n_hav = int(still.sum())
    if n_hav:
        H = haversine_matrix(c["latitude"], c["longitude"]) * CIRCUITY
        S[still] = H[still]
    np.fill_diagonal(S, 0.0)

    out = pd.DataFrame(np.round(S, 3), index=m.index, columns=m.columns)
    out.index.name = "district_id"
    print(f"Invalid directed entries: {int(invalid.sum())} | from reverse: {n_mirror} | Haversine x {CIRCUITY}: {n_hav}")
    return out

In [ ]:
def fix_island_districts(matrix: pd.DataFrame, coords: pd.DataFrame) -> pd.DataFrame:
    """Districts with no road link to the mainland (ISLAND_STATES): OSRM snaps them to unrelated
    road points. Replace their rows/columns with Haversine x CIRCUITY."""
    c = coords.set_index("district_id").loc[matrix.index]
    idx = np.where(c["state"].str.contains("|".join(ISLAND_STATES)).to_numpy())[0]
    print("Replacing:", [(int(matrix.index[i]), c["district_name"].iloc[i]) for i in idx])

    A = matrix.to_numpy(float).copy()
    H = haversine_matrix(c["latitude"], c["longitude"]) * CIRCUITY
    for i in idx:
        A[i, :] = H[i, :]
        A[:, i] = H[:, i]
    np.fill_diagonal(A, 0.0)

    out = pd.DataFrame(np.round(A, 3), index=matrix.index, columns=matrix.columns)
    out.index.name = "district_id"
    return out

In [ ]:
def intra_district_distance(n_districts: int) -> float:
    """Expected road km between a random supplier and a random requester in the same district.
    Average district area = INDIA_AREA_KM2 / n_districts, modelled as a disc of equal area.
    Mean distance between two uniform random points in a disc of radius R is 128R/(45*pi);
    multiplied by CIRCUITY to approximate road distance."""
    area = INDIA_AREA_KM2 / n_districts
    radius = np.sqrt(area / np.pi)
    return round(128 * radius / (45 * np.pi) * CIRCUITY, 3)

In [ ]:
def set_intra_district_distance(matrix: pd.DataFrame) -> pd.DataFrame:
    """Replace the zero diagonal with a constant intra-district distance (INTRA_DISTRICT_KM,
    or the value from intra_district_distance() when it is None)."""
    value = INTRA_DISTRICT_KM if INTRA_DISTRICT_KM is not None else intra_district_distance(len(matrix))
    A = matrix.to_numpy(float).copy()
    np.fill_diagonal(A, value)
    out = pd.DataFrame(A, index=matrix.index, columns=matrix.columns)
    out.index.name = "district_id"
    print(f"Diagonal set to {value} km")
    return out

In [ ]:
def validate_matrix(matrix: pd.DataFrame, coords: pd.DataFrame) -> None:
    """Assert: NxN, labels match, no NaN/inf/negatives, constant diagonal, symmetric.
    Warn on off-diagonal zeros and on road < Haversine pairs."""
    n = len(matrix)
    A = matrix.to_numpy(float)
    assert matrix.shape == (n, n), f"Shape {matrix.shape} != ({n}, {n})"
    assert (matrix.index == matrix.columns).all(), "Row/column labels differ"
    assert not matrix.isna().any().any() and np.isfinite(A).all(), "NaN/inf present"
    assert (A >= 0).all(), "Negative distances"
    assert np.ptp(np.diag(A)) == 0, "Diagonal is not constant"
    assert np.abs(A - A.T).max() == 0, "Not symmetric"

    eye = np.eye(n, dtype=bool)
    zeros = int(((A == 0) & ~eye).sum())
    if zeros:
        print(f"WARNING: {zeros} off-diagonal zeros (identical coordinates)")
    c = coords.set_index("district_id").loc[matrix.index]
    H = haversine_matrix(c["latitude"], c["longitude"])
    short = int(((A < 0.95 * H) & (H > 50)).sum() // 2)
    if short:
        print(f"WARNING: {short} pairs have road distance < 0.95 x Haversine (snapping artifacts)")
    print(f"Valid: {n}x{n}, diagonal {A[0, 0]} km, max {A.max():.1f} km")

## 5. Export (D### ids, two sheets)

In [ ]:
def export_xlsx(matrix: pd.DataFrame, coords: pd.DataFrame, path: str = OUT_XLSX) -> pd.DataFrame:
    """Write sheet "districts" (district_id, name, state) and sheet "distances" (D### x D### road km).
    Returns the old-id -> D### mapping (also saved to OUT_MAP_CSV)."""
    c = coords.set_index("district_id").loc[matrix.index].reset_index().sort_values("district_id").reset_index(drop=True)
    c["new_id"] = [f"D{i:03d}" for i in range(1, len(c) + 1)]
    c["state"] = c["state"].str.replace(r"\s*\([A-Za-z]{2}\)\s*$", "", regex=True).str.strip()

    id_map = dict(zip(c["district_id"], c["new_id"]))
    m = matrix.loc[c["district_id"], c["district_id"]]
    m.index = m.index.map(id_map)
    m.columns = m.columns.map(id_map)

    wb = Workbook()
    wb._named_styles["Normal"].font = Font(name="Arial", size=10)
    ws = wb.active
    ws.title = "districts"
    ws.append(["district_id", "name", "state"])
    for r in c.itertuples():
        ws.append([r.new_id, r.district_name, r.state])
    for cell in ws[1]:
        cell.font = Font(name="Arial", bold=True)
    ws.column_dimensions["A"].width = 13
    ws.column_dimensions["B"].width = 32
    ws.column_dimensions["C"].width = 38
    ws.freeze_panes = "A2"

    wd = wb.create_sheet("distances")
    wd.append(["district_id"] + list(m.columns))
    for label, row in zip(m.index, m.to_numpy(float).tolist()):
        wd.append([label] + row)
    for cell in wd[1]:
        cell.font = Font(name="Arial", bold=True)
    for r in range(2, wd.max_row + 1):
        wd.cell(r, 1).font = Font(name="Arial", bold=True)
    wd.freeze_panes = "B2"
    wb.save(path)

    mapping = pd.DataFrame({"old_id": c["district_id"], "district_id": c["new_id"]})
    mapping.to_csv(OUT_MAP_CSV, index=False)
    print(f"Wrote {path}: districts {len(c)}, distances {m.shape}")
    return mapping

## Run the full pipeline

In [ ]:
def run_pipeline():
    districts = scrape_districts()
    print(f"Scraped {len(districts)} districts")
    coords = geocode_districts(districts)
    raw = build_osrm_matrix(coords)
    cleaned = clean_matrix(raw, coords)
    fixed = fix_island_districts(cleaned, coords)
    final = set_intra_district_distance(fixed)
    validate_matrix(final, coords)
    export_xlsx(final, coords)
    return final


final_matrix = run_pipeline()